In [ ]:
import numpy as np 
import librosa 
import matplotlib.pyplot as plt 

In [ ]:
y, sr = librosa.load("../data/test_piano.mp3", sr=None, mono=True, duration=30)
#par defaut, librosa utilise un echantillonnage de 22050 Hz, on utilise None pour avoir l'echantillonnage de base du fichier.
#y = signal

In [ ]:
print("Type = ", type(y)) 
print("Type des nombres :", y.dtype)
print("Forme :", y.shape)
print("Fréquence d'échantillonnage :", sr, "Hz")
print("10 premiers échantillons :", y[:10])
print("Min :", y.min(), "| Max :", y.max())

In [ ]:
print("duration = ", len(y) / sr, "s")

In [ ]:
from IPython.display import Audio

In [ ]:
Audio(y, rate = sr) 

In [ ]:
t = np.arange(len(y)) / sr # cree tableau avec numero d'echantillon et divise chacun par sample rate pour avoir la seconde associé
print(t[:10])

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(t, y, linewidth=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Wave shape : first 30 seconds") 

In [ ]:
librosa.display.waveshow(y, sr=sr) 

In [ ]:
# each notes produces a strong peak when the hammer hits the rope, then a progressive decrease, the rope vibrates more and more weakly

In [ ]:
start = int(5.0 * sr)
end = start + int(0.02 * sr)

plt.figure(figsize=(14, 4))
plt.plot(t[start:end], y[start:end])
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("20ms window zoom starting at 5sec") 
plt.show()

In [ ]:
# we can observe a fundamental frequency to each are added harmonic freqencies, they are all tangled, we need a tool to untangle them
# -> Fourier tranform

In [ ]:
# SPECTROGRAM

In [ ]:
n_fft = 2048 # size of each window
hop_length = 512 # offset between 2 windows

# Short-time Fourier transform
S = librosa.stft(y, n_fft = n_fft, hop_length = hop_length) 

print("Shape = ", S.shape)
print("Type = ", S.dtype)
print("Frequence resolution = ", sr / n_fft, "Hz") 
print("Time resolution = ", hop_length / sr * 100, "ms") 

In [ ]:
# 1025 frequency slices 
# FFT on 2048 samples give 2048/2 + 1 = 1025 useful samples (Nyquist-Shannon) 
# 2584 columns = 1 by window = 30s / 11.6ms
# Complex numbers : module says how much a freqency is present & phase says what's the status of the oscillation at the specific moment.

In [ ]:
# Turn into decibels (dB)
S_dB = librosa.amplitude_to_db(np.abs(S), ref = np.max)
# abs compute the module of each complex number : we keep the intensity
# turn into dB wrt the maximum: highest value is 0, everything else is negative

In [ ]:
import librosa.display

fig, ax = plt.subplots(figsize=(14,4))
img = librosa.display.specshow(
    S_dB,
    sr=sr,
    hop_length=hop_length,
    x_axis="time",
    y_axis="log",
    ax=ax
)
fig.colorbar(img, ax=ax, format="%+2.0f dB")
ax.set_title("Spectogram (STFT), logarithmic frequencies axis") 
plt.show()

In [ ]:
# vertical lines = when the hammer hits
# STFT cuts frequencies in fixed-size slices (~21.5Hz)
# Sol2 = 98Hz and Ré2 = 73Hz. Between 64 and 128 Hz, a 21.5Hz slice contains 4 to 6 half-tones (not accurate enough to identify the notes)

In [ ]:
# CQT method 

fmin = librosa.note_to_hz("A0") # convert note into frequency

C = librosa.cqt(
    y,
    sr=sr,
    hop_length=hop_length,
    fmin=fmin,
    n_bins=88, # 88 piano keys
    bins_per_octave=12, # 12 slices per octave (12 half tones)
)
C_dB = librosa.amplitude_to_db(np.abs(C), ref=np.max)

fig, ax = plt.subplots(figsize=(14,4))
img = librosa.display.specshow(
    C_dB,
    sr=sr,
    hop_length=hop_length,
    x_axis="time",
    y_axis="cqt_note",
    fmin=fmin,
    bins_per_octave=12,
    ax=ax,
)

fig.colorbar(img, ax=ax, format="%+2.0f dB")
ax.set_title("CQT : one line per piano key")
plt.show()

In [ ]:
# CQT allows to match a line to a note.
# from 0 to 1.5s : no sound
# Between C2 and C3, two notes alternate : low sounds of left hand (Sol2/G2 & Ré2/D2)
# Around C4, right after each low notes, a lot of bright lines: chords played by left hand (left hand jumps from low notes to chords). The bright lines change slightly after each alternance because of the harmony.
# Between 10 and 12s: new lines between C5 and C6: right hand melody
# Purple lines are the harmonies of each note
# Around 15s : black vertical line = micro cut in the sound recording